# Centenarian Phenotype Model — quickstart (toy/synthetic data only)

Runs the deployable `centenarian_phenotype` package on **made-up profiles** to show the API.
No real or person-level data is used.

**Claim boundary:** this is an *evidence-weighted similarity* score with a derived,
calibration-pending Naive Bayes posterior and a *preliminary* mortality signal — **not** a
validated centenarian-prediction or medical tool. See `MODEL_CARD.md`.

Install: `pip install -e .` from the repo root.

In [ ]:
import centenarian_phenotype as cp
print('package version', cp.__version__)
print('model versions', cp.MODEL_VERSIONS)

## Tier 1 — teaser quiz (made-up answers: option index per question)

In [ ]:
r1 = cp.score(1, {'q_physical_activity': 0, 'q_diet': 0, 'q_smoking': 0,
                  'q_social': 0, 'q_purpose': 0, 'q_sleep': 0})
print('similarity score_pct:', r1['score_pct'])
print('class posteriors    :', r1['class_posteriors'])
print('model depth / resp / evidence conf:',
      r1['model_depth_pct'], r1['response_completeness_pct'], r1['evidence_confidence_pct'])
print('next best action    :', r1['next_best_data_action'])

## Tier 3 — raw labs via mappers + a DNAm clock, with a relative-longevity context

In [ ]:
# raw lab panel -> alignments (referenced cut-offs; no invented math)
panel = cp.map_panel({'hdl_cholesterol': 62, 'triglycerides': 90, 'glucose': 95,
                      'hba1c': 5.3, 'c_reactive_protein': 0.4, 'body_mass_index': 22},
                     sex='F', age=72)
clinical = panel['clinical']

# a biological-age clock reading -> alignment (GrimAge 60 at chronological age 72)
from centenarian_phenotype.clocks import compute_panel, to_clinical_alignments
clocks = compute_panel({'grimage_2019': {'value': 60, 'chronological_age': 72}}, sex='F')
clinical.update(to_clinical_alignments(clocks))

r3 = cp.score(3, {'q_smoking': 0, 'q_loneliness': 0},
              clinical=clinical,
              context={'country': 'Japan', 'sex': 'F', 'age': 72, 'bio_age_delta': -12})
print('score_pct           :', r3['score_pct'])
print('domains             :', list(r3['domain_scores']))
lc = r3['longevity_context']
print('population baseline :', lc['population_baseline']['p_reach_target'], '(P reach 100 | alive 65)')
print('calibrated 10-yr mortality:', lc['calibrated_mortality'])

## Validation harness (synthetic self-test)
From the repo root: `python scripts/validation/validate.py --synthetic 4000 --out reports/demo`
— runs the AUC / calibration / ablation engine on a generated cohort (no real data).